# 2026 Housing EIS — TDM Data Source Inventory & Snapshot

This notebook identifies every external and local data source used across all scripts in the 2026 Housing EIS Travel Demand Model analysis, then creates a reproducibility snapshot.

## What it does
1. **Fetches external data** — ArcGIS feature services, LakeTahoeInfo JSON APIs, SQL Server tables
2. **Archives spatial layers** as feature classes in `TDM_DataInventory.gdb`
3. **Archives tabular data** as CSVs in `TDM_Inputs/`
4. **Copies local curated files** (Lookup_Lists, inputs, raw data) to the archive
5. **Copies scenario config JSONs** to the archive
6. **Writes `TDM_DataInventory_Record.csv`** — a master catalogue with source, script, and archive location for every data source

**Purpose:** If authoritative data changes, this snapshot allows the analysis to be re-run from a fixed point in time.

**Archive location:** `F:/GIS/GIS_DATA/AnalysisDataArchives/CultivatingCommunityTDM/`

**Run from:** The `TravelDemandModel/2026_Housing_EIS/` directory (where this notebook lives)

---
## Coverage
| Group | Count | Type |
|-------|-------|------|
| ArcGIS feature services (spatial) | 8 | → GDB feature classes |
| ArcGIS feature services (tabular) | 4 | → CSV |
| LakeTahoeInfo JSON APIs | 5 | → CSV |
| SQL Server tables | 3 | → CSV |
| Local curated files | 24+ | → copied to archive |
| Scenario config JSONs | varies | → copied to archive |

## 1 — Setup

In [12]:
import sys
import os
import re
import shutil
import traceback
import json
import datetime
from pathlib import Path

import pandas as pd
import arcpy
from arcgis.features import FeatureLayer
from sqlalchemy.engine import URL
from sqlalchemy import create_engine

# ── Configurable year parameters ───────────────────────────────────────────────
PARCEL_YEAR  = 2022          # Year queried from Existing Development service
CAMP_YEAR    = 2022          # Year queried from campground visits service
SCHOOL_YEAR  = '2021-2022'   # School year queried from enrollment service

# ── Archive output paths ───────────────────────────────────────────────────────
ARCHIVE_ROOT   = Path('F:/GIS/GIS_DATA/AnalysisDataArchives/CultivatingCommunityTDM')
GDB_PATH       = ARCHIVE_ROOT / 'TDM_DataInventory.gdb'
CSV_OUT_DIR    = ARCHIVE_ROOT / 'TDM_Inputs'
CONFIGS_OUT    = ARCHIVE_ROOT / 'TDM_Configs'
INVENTORY_PATH = ARCHIVE_ROOT / 'TDM_DataInventory_Record.csv'

# ── Source paths relative to this notebook (2026_Housing_EIS/) ────────────────
NB_DIR           = Path().absolute()
BASE_LOOKUP_DIR  = NB_DIR / 'Base'    / 'scripts' / 'Lookup_Lists'
BASE_RAW_DIR     = NB_DIR / 'Base'    / 'data'    / 'raw_data'
FCST_LOOKUP_DIR  = NB_DIR / 'Forecast' / 'scripts' / 'Lookup_Lists'
FCST_INPUTS_DIR  = NB_DIR / 'Forecast' / 'data'    / 'inputs'
FCST_CONFIGS_DIR = NB_DIR / 'Forecast' / 'configs_final'

# ── Service URL roots ──────────────────────────────────────────────────────────
TRPA_BASE   = 'https://maps.trpa.org/server/rest/services'
LTINFO_BASE = 'https://www.laketahoeinfo.org/WebServices'
LTINFO_KEY  = 'e17aeb86-85e3-4260-83fd-a2b32501c476'

SNAPSHOT_DATE = datetime.datetime.now().strftime('%Y-%m-%d')
SNAPSHOT_TS   = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

print(f'Notebook directory : {NB_DIR}')
print(f'Archive root       : {ARCHIVE_ROOT}')
print(f'GDB path           : {GDB_PATH}')
print(f'CSV output dir     : {CSV_OUT_DIR}')
print(f'Snapshot timestamp : {SNAPSHOT_TS}')

Notebook directory : c:\Users\amcclary\Documents\GitHub\Transportation\TravelDemandModel\2026_Housing_EIS
Archive root       : F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM
GDB path           : F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_DataInventory.gdb
CSV output dir     : F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_Inputs
Snapshot timestamp : 2026-05-12 13:40:01


## 2 — Helper Functions

In [13]:
# ── SQL connection factory (mirrors utils.py) ──────────────────────────────────
def get_sql_engine(db='sde'):
    db_user     = os.environ.get('DB_USER')
    db_password = os.environ.get('DB_PASSWORD')
    driver      = 'ODBC Driver 17 for SQL Server'
    servers     = {'sde': 'sql12', 'sde_tabular': 'sql12', 'tahoebmpsde': 'sql14'}
    server      = servers.get(db.lower(), 'sql12')
    conn_str    = (f'DRIVER={driver};SERVER={server};DATABASE={db};'
                   f'UID={db_user};PWD={db_password}')
    conn_url = URL.create('mssql+pyodbc', query={'odbc_connect': conn_str})
    return create_engine(conn_url)


# ── SQL query that strips geometry columns (mirrors utils.py) ──────────────────
def read_sql_no_geom(query, engine):
    with engine.connect() as conn:
        if not re.search(r'SELECT\s+\*', query, re.IGNORECASE):
            return pd.read_sql(query, conn)
        match = re.search(r'FROM\s+([\w\.\[\]]+)', query, re.IGNORECASE)
        if not match:
            return pd.read_sql(query, conn)
        table_ref = match.group(1).replace('[', '').replace(']', '')
        parts     = table_ref.split('.')
        schema    = parts[-2] if len(parts) >= 2 else None
        table     = parts[-1]
        s_clause  = f"TABLE_SCHEMA = '{schema}' AND " if schema else ''
        col_q     = (f'SELECT COLUMN_NAME FROM INFORMATION_SCHEMA.COLUMNS '
                     f'WHERE {s_clause}TABLE_NAME = \'{table}\' '
                     f"AND DATA_TYPE NOT IN ('geometry', 'geography') "
                     f'ORDER BY ORDINAL_POSITION')
        df_cols   = pd.read_sql(col_q, conn)
        if df_cols.empty:
            return pd.read_sql(query, conn)
        col_list  = ', '.join(f'[{c}]' for c in df_cols['COLUMN_NAME'])
        new_query = re.sub(r'SELECT\s+\*', f'SELECT {col_list}', query, flags=re.IGNORECASE)
        return pd.read_sql(new_query, conn)


# ── Fix SEDF geometry after arcgis/geopandas dtype conflict ───────────────────
def fix_sedf_geometry(sdf, geom_col='SHAPE'):
    from arcgis.geometry import Geometry
    if geom_col not in sdf.columns:
        return sdf
    result = sdf.copy()
    def _coerce(g):
        if g is None:
            return None
        if isinstance(g, Geometry):
            return g
        if hasattr(g, '__geo_interface__'):
            return Geometry(g.__geo_interface__)
        return g
    result[geom_col] = [_coerce(g) for g in result[geom_col].to_numpy(dtype=object)]
    return result


# ── Snapshot a spatial feature service to the archive GDB ─────────────────────
def snapshot_spatial_to_gdb(src, log):
    name   = src['name']
    url    = src['url']
    q      = src.get('query')
    fc     = src['fc_name']
    out_fc = os.path.join(str(GDB_PATH), fc)
    print(f'  [{src["id"]}] {name}')
    if q:
        print(f'    Query: {q}')
    try:
        fl = FeatureLayer(url)
        if q:
            sdf = fl.query(where=q).sdf
        else:
            sdf = pd.DataFrame.spatial.from_layer(fl)
        sdf = fix_sedf_geometry(sdf)
        if arcpy.Exists(out_fc):
            arcpy.management.Delete(out_fc)
        sdf.spatial.to_featureclass(out_fc)
        n = len(sdf)
        print(f'    OK — {n:,} features written to {fc}')
        log.append({'source_id': src['id'], 'status': 'SUCCESS',
                    'record_count': n, 'archive_location': out_fc, 'error_message': ''})
    except Exception as e:
        print(f'    ERROR: {e}')
        log.append({'source_id': src['id'], 'status': 'ERROR',
                    'record_count': 0, 'archive_location': out_fc, 'error_message': str(e)})


# ── Snapshot a tabular feature service to CSV ──────────────────────────────────
def snapshot_tabular_to_csv(src, log):
    name     = src['name']
    url      = src['url']
    q        = src.get('query')
    csv_name = src['csv_name']
    out_path = CSV_OUT_DIR / csv_name
    print(f'  [{src["id"]}] {name}')
    try:
        fl = FeatureLayer(url)
        result = fl.query(where=q) if q else fl.query()
        df = pd.DataFrame([f.attributes for f in result.features])
        df.to_csv(out_path, index=False)
        print(f'    OK — {len(df):,} records → {csv_name}')
        log.append({'source_id': src['id'], 'status': 'SUCCESS',
                    'record_count': len(df), 'archive_location': str(out_path), 'error_message': ''})
    except Exception as e:
        print(f'    ERROR: {e}')
        log.append({'source_id': src['id'], 'status': 'ERROR',
                    'record_count': 0, 'archive_location': str(out_path), 'error_message': str(e)})


# ── Snapshot a LakeTahoeInfo JSON API to CSV ───────────────────────────────────
def snapshot_ltinfo_to_csv(src, log):
    name     = src['name']
    url      = src['url']
    csv_name = src['csv_name']
    out_path = CSV_OUT_DIR / csv_name
    print(f'  [{src["id"]}] {name}')
    try:
        df = pd.read_json(url)
        df.to_csv(out_path, index=False)
        print(f'    OK — {len(df):,} records → {csv_name}')
        log.append({'source_id': src['id'], 'status': 'SUCCESS',
                    'record_count': len(df), 'archive_location': str(out_path), 'error_message': ''})
    except Exception as e:
        print(f'    ERROR: {e}')
        log.append({'source_id': src['id'], 'status': 'ERROR',
                    'record_count': 0, 'archive_location': str(out_path), 'error_message': str(e)})


# ── Snapshot a SQL table to CSV ────────────────────────────────────────────────
def snapshot_sql_to_csv(src, log):
    name     = src['name']
    db       = src['db']
    query    = src['query']
    csv_name = src['csv_name']
    out_path = CSV_OUT_DIR / csv_name
    print(f'  [{src["id"]}] {name}')
    print(f'    Query: {query}')
    try:
        engine = get_sql_engine(db)
        df = read_sql_no_geom(query, engine)
        df.to_csv(out_path, index=False)
        print(f'    OK — {len(df):,} records → {csv_name}')
        log.append({'source_id': src['id'], 'status': 'SUCCESS',
                    'record_count': len(df), 'archive_location': str(out_path), 'error_message': ''})
    except Exception as e:
        print(f'    ERROR: {e}')
        log.append({'source_id': src['id'], 'status': 'ERROR',
                    'record_count': 0, 'archive_location': str(out_path), 'error_message': str(e)})


# ── Copy a local file to the archive ──────────────────────────────────────────
def copy_local_file(src, log):
    name      = src['name']
    src_path  = src['source_path']
    dest_name = src['dest_name']
    out_path  = CSV_OUT_DIR / dest_name
    print(f'  [{src["id"]}] {name}')
    if not Path(src_path).exists():
        msg = f'Source file not found: {src_path}'
        print(f'    SKIP — {msg}')
        log.append({'source_id': src['id'], 'status': 'SKIP',
                    'record_count': 0, 'archive_location': str(out_path), 'error_message': msg})
        return
    try:
        shutil.copy2(src_path, out_path)
        row_count = sum(1 for _ in open(src_path)) - 1  # approx rows minus header
        print(f'    OK — copied to {dest_name}')
        log.append({'source_id': src['id'], 'status': 'SUCCESS',
                    'record_count': max(row_count, 0), 'archive_location': str(out_path), 'error_message': ''})
    except Exception as e:
        print(f'    ERROR: {e}')
        log.append({'source_id': src['id'], 'status': 'ERROR',
                    'record_count': 0, 'archive_location': str(out_path), 'error_message': str(e)})


print('Helper functions defined.')

Helper functions defined.


## 3 — Data Source Definitions

All sources are catalogued here. Each group corresponds to one fetch/copy section below.

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# GROUP 1 — Spatial ArcGIS Feature Services → written as GDB feature classes
# ══════════════════════════════════════════════════════════════════════════════
SPATIAL_SERVICES = [
    {
        'id': 'DS_SPATIAL_001',
        'name': 'Existing Development Parcels',
        'description': (
            'Parcel-level existing development data — residential units, tourist accommodation '
            'units, commercial floor area, jurisdiction, land use, ownership. The foundational '
            'spatial layer for all parcel-level analysis.'
        ),
        'url': f'{TRPA_BASE}/Existing_Development/MapServer/2',
        'query': f'Year = {PARCEL_YEAR}',
        'fc_name': 'Existing_Dev_Parcels',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': f'Filtered to Year = {PARCEL_YEAR}',
    },
    {
        'id': 'DS_SPATIAL_002',
        'name': 'Traffic Analysis Zones (TAZ)',
        'description': (
            'TAZ polygon boundaries. Used for all spatial aggregation of parcels to '
            'socioeconomic model inputs (households, persons, employment, visitors).'
        ),
        'url': f'{TRPA_BASE}/Transportation_Planning/MapServer/6',
        'query': None,
        'fc_name': 'TAZ',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Forecast/scripts/Additional_Jobs.ipynb',
            'Forecast/scripts/Job_Distribution.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'All features',
    },
    {
        'id': 'DS_SPATIAL_003',
        'name': 'Census Block Groups',
        'description': (
            'Census block group polygons. Used for demographic crosswalks — household size, '
            'occupancy rates, income proportions from ACS. Script filters to YEAR=2020 '
            'and TRPAID length=16 for block-group level features.'
        ),
        'url': f'{TRPA_BASE}/Demographics/MapServer/27',
        'query': None,
        'fc_name': 'Block_Groups',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'All years/levels. Filtered to YEAR=2020 and len(TRPAID)==16 in script.',
    },
    {
        'id': 'DS_SPATIAL_004',
        'name': 'Census Tracts',
        'description': (
            'Census tract polygons. Used for demographic data aggregation and '
            'employment allocation overlap calculations.'
        ),
        'url': f'{TRPA_BASE}/Demographics/MapServer/28',
        'query': None,
        'fc_name': 'Census_Tracts',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Forecast/scripts/Additional_Jobs.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'All features',
    },
    {
        'id': 'DS_SPATIAL_005',
        'name': 'Campgrounds',
        'description': (
            'Campground point locations. Spatially joined to TAZ to distribute '
            'campground capacity and overnight visitors into the travel model input.'
        ),
        'url': f'{TRPA_BASE}/Recreation/MapServer/1',
        'query': "RECREATION_TYPE='Campground'",
        'fc_name': 'Campgrounds',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': "Filtered to RECREATION_TYPE='Campground'",
    },
    {
        'id': 'DS_SPATIAL_006',
        'name': 'Tourist Accommodation Occupancy Zones',
        'description': (
            'Occupancy zone polygons. Used to spatially assign occupancy rates to '
            'tourist accommodation parcels via IDW interpolation within zones.'
        ),
        'url': f'{TRPA_BASE}/Transportation_Planning/MapServer/15',
        'query': None,
        'fc_name': 'Occupancy_Zones',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'All features',
    },
    {
        'id': 'DS_SPATIAL_007',
        'name': 'Schools (Spatial)',
        'description': (
            'School location points. Used for spatial crosswalk to TAZ to build '
            'the school enrollment model input file.'
        ),
        'url': f'{TRPA_BASE}/Transportation_Planning/MapServer/16',
        'query': None,
        'fc_name': 'Schools',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'All features',
    },
    {
        'id': 'DS_SPATIAL_008',
        'name': 'Town Centers',
        'description': (
            'Town center polygon boundaries. Used to identify parcels eligible for the '
            'town center housing pool and to compute employment for town center housing '
            'units added in the forecast.'
        ),
        'url': f'{TRPA_BASE}/Planning/FeatureServer/3',
        'query': None,
        'fc_name': 'Town_Centers',
        'scripts_used_in': [
            'Forecast/scripts/Additional_Jobs.ipynb',
            'Forecast/scripts/Job_Distribution.ipynb',
        ],
        'notes': 'All features',
    },
]

print(f'Spatial services defined: {len(SPATIAL_SERVICES)}')

Spatial services defined: 8


In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# GROUP 2 — Tabular ArcGIS Feature Services → CSV (no geometry)
# ══════════════════════════════════════════════════════════════════════════════
TABULAR_SERVICES = [
    {
        'id': 'DS_TABULAR_001',
        'name': 'VHR Parcel List',
        'description': (
            'List of parcels registered as Vacation Home Rentals (VHRs). '
            'Used to flag VHR parcels and assign VHR-specific occupancy rates '
            'in the visitor accommodation model.'
        ),
        'url': f'{TRPA_BASE}/VHR/MapServer/0',
        'query': None,
        'csv_name': 'VHR_List.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'Non-spatial table. All records.',
    },
    {
        'id': 'DS_TABULAR_002',
        'name': 'Campground Visitation Records',
        'description': (
            'Annual campground visitation records (sites sold, occupancy rates by campground). '
            'Joined to campground spatial layer to compute overnight visitor zonal data.'
        ),
        'url': f'{TRPA_BASE}/Transportation_Planning/MapServer/14',
        'query': f'Year = {CAMP_YEAR}',
        'csv_name': 'Campground_Visits.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': f'Filtered to Year = {CAMP_YEAR}',
    },
    {
        'id': 'DS_TABULAR_003',
        'name': 'Tourist Accommodation Occupancy Rates (Service)',
        'description': (
            'Observed occupancy rates by tourist accommodation type and zone. '
            'Used as inputs to IDW interpolation across occupancy zones. '
            'Note: base_year_data_engineering.ipynb reads the local occupancy_rates_raw.csv '
            'instead of this live service.'
        ),
        'url': f'{TRPA_BASE}/Transportation_Planning/MapServer/13',
        'query': None,
        'csv_name': 'Occupancy_Rates_Svc.csv',
        'scripts_used_in': [
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'All records. Live service version of occupancy_rates_raw.csv.',
    },
    {
        'id': 'DS_TABULAR_004',
        'name': 'School Enrollment',
        'description': (
            'School enrollment counts by school and TAZ for the base year. '
            'Used to build the SchoolEnrollment.csv model input file.'
        ),
        'url': f'{TRPA_BASE}/Transportation_Planning/MapServer/17',
        'query': f"Year = '{SCHOOL_YEAR}'",
        'csv_name': 'School_Enrollment_Svc.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': f"Filtered to Year = '{SCHOOL_YEAR}'",
    },
]

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 3 — LakeTahoeInfo JSON Web Services → CSV
# ══════════════════════════════════════════════════════════════════════════════
LTINFO_SOURCES = [
    {
        'id': 'DS_LTINFO_001',
        'name': 'Parcel IPES Scores',
        'description': (
            'Individual Parcel Evaluation System (IPES) scores for all parcels. '
            'Used to determine development eligibility: IPES > 0 required for most '
            'jurisdictions, IPES > 726 required for Placer County.'
        ),
        'url': f'{LTINFO_BASE}/GetParcelIPESScores/JSON/{LTINFO_KEY}',
        'csv_name': 'IPES_Scores_LTInfo.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'All parcels with IPES scores',
    },
    {
        'id': 'DS_LTINFO_002',
        'name': 'Parcel Land Capability Values',
        'description': (
            'Land Capability Value (LCV) assignments for all parcels. '
            'Used for environmental suitability assessment in development eligibility.'
        ),
        'url': f'{LTINFO_BASE}/GetParcelsByLandCapability/JSON/{LTINFO_KEY}',
        'csv_name': 'Land_Capability_LTInfo.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'All parcels with land capability assignments',
    },
    {
        'id': 'DS_LTINFO_003',
        'name': 'All Parcels Status (LTInfo)',
        'description': (
            'All parcel records from LTInfo including the RetiredFromDevelopment flag. '
            'Used to mark parcels that have been permanently retired from '
            'development eligibility.'
        ),
        'url': f'{LTINFO_BASE}/GetAllParcels/JSON/{LTINFO_KEY}',
        'csv_name': 'All_Parcels_LTInfo.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'Includes RetiredFromDevelopment field. Same endpoint queried twice in script.',
    },
    {
        'id': 'DS_LTINFO_004',
        'name': 'Banked Development Rights',
        'description': (
            'Records of banked development rights by parcel. Used to track '
            'development rights that have been banked (held in reserve).'
        ),
        'url': f'{LTINFO_BASE}/GetBankedDevelopmentRights/JSON/{LTINFO_KEY}',
        'csv_name': 'Banked_Dev_Rights_LTInfo.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'All banked development right records',
    },
    {
        'id': 'DS_LTINFO_005',
        'name': 'Transacted and Banked Development Rights',
        'description': (
            'Records of transacted and banked development rights by parcel. '
            'Used to identify parcels involved in development rights transfers.'
        ),
        'url': f'{LTINFO_BASE}/GetTransactedAndBankedDevelopmentRights/JSON/{LTINFO_KEY}',
        'csv_name': 'Transacted_Banked_LTInfo.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'All transacted and banked development right records',
    },
]

# ══════════════════════════════════════════════════════════════════════════════
# GROUP 4 — SQL Server Tables → CSV
# ══════════════════════════════════════════════════════════════════════════════
SQL_SOURCES = [
    {
        'id': 'DS_SQL_001',
        'name': 'Permissible Uses (Zoning)',
        'description': (
            'Zoning permissible use table — maps Zoning_ID to Use_Type '
            '(Single Family, Multiple Family, Commercial, Tourist Accommodation) and density. '
            'Used to classify every parcel by housing zoning type and compute maximum '
            'allowed residential units per parcel.'
        ),
        'db': 'sde',
        'query': 'SELECT * FROM sde.SDE.PermissibleUses',
        'csv_name': 'PermissibleUses_SQL.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'SQL Server: sql12 / sde database. No geometry columns.',
    },
    {
        'id': 'DS_SQL_002',
        'name': 'Special Designation (Zoning)',
        'description': (
            'Special zoning designation table — identifies Transfer and Receive zones for '
            'development rights transfers. Used to flag parcels in receiving zones '
            'that are eligible for bonus unit pool development.'
        ),
        'db': 'sde',
        'query': 'SELECT * FROM sde.SDE.Special_Designation',
        'csv_name': 'Special_Designation_SQL.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'SQL Server: sql12 / sde database. No geometry columns.',
    },
    {
        'id': 'DS_SQL_003',
        'name': 'Parcel History Attributed — 2022',
        'description': (
            'Fully attributed parcel history snapshot for 2022 — includes address, '
            'plan name, zoning description, town center assignment, VHR flag, '
            'and other administrative fields. Used to back-fill attribute fields '
            'into the 2026 base parcel dataset where fields are incomplete.'
        ),
        'db': 'sde',
        'query': 'SELECT * FROM sde.SDE.Parcel_History_Attributed WHERE YEAR = 2022',
        'csv_name': 'Parcel_History_2022_SQL.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'SQL Server: sql12 / sde database. Geometry columns stripped by read_sql_no_geom().',
    },
]

print(f'Tabular services : {len(TABULAR_SERVICES)}')
print(f'LTInfo sources   : {len(LTINFO_SOURCES)}')
print(f'SQL sources      : {len(SQL_SOURCES)}')

Tabular services : 4
LTInfo sources   : 5
SQL sources      : 3


In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# GROUP 5 — Local Curated Files → copied to archive
# ══════════════════════════════════════════════════════════════════════════════
LOCAL_FILES = [
    # ── Base/scripts/Lookup_Lists ──────────────────────────────────────────────
    {
        'id': 'DS_LOCAL_001',
        'name': 'Development Changes 2022-2025',
        'description': (
            'Hand-curated list of known unit changes (new permits, demolitions, conversions) '
            'that occurred between 2022 and the 2026 base year. Applied to update the '
            'parcel dataset from the 2022 snapshot to current conditions.'
        ),
        'source_path': BASE_LOOKUP_DIR / 'development_2022_2025.csv',
        'dest_name': 'development_2022_2025.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'Curated lookup list — Base/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_002',
        'name': 'Tourist Accommodation Unit Type Lookup',
        'description': (
            'Maps parcel APNs to TAU type categories (HotelMotel, Resort, Casino, etc.). '
            'Controls which occupancy rate curve is applied to each tourist parcel.'
        ),
        'source_path': BASE_LOOKUP_DIR / 'lookup_tau_type.csv',
        'dest_name': 'lookup_tau_type.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'Curated lookup list — Base/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_003',
        'name': 'Closed Tourist Parcels',
        'description': (
            'List of tourist accommodation parcels that are known to be permanently '
            'closed. These are removed from the tourist accommodation unit inventory '
            'before modeling.'
        ),
        'source_path': BASE_LOOKUP_DIR / 'closed_tourist_parcels.csv',
        'dest_name': 'closed_tourist_parcels.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'Curated lookup list — Base/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_004',
        'name': 'Income Census Codes',
        'description': (
            'Crosswalk between ACS census income category codes and model income categories '
            '(low/medium/high). Used to aggregate ACS block-group income data to '
            'the three model income tiers.'
        ),
        'source_path': BASE_LOOKUP_DIR / 'income_census_codes.csv',
        'dest_name': 'income_census_codes.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
            'Base/scripts/Archive/notebook.ipynb',
        ],
        'notes': 'Curated lookup list — Base/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_005',
        'name': 'Work-From-Home Rate by TAZ (Base)',
        'description': (
            'TAZ-level work-from-home rates used to adjust employment numbers '
            'for remote work patterns in the base year data engineering.'
        ),
        'source_path': BASE_LOOKUP_DIR / 'taz_work_from_home.csv',
        'dest_name': 'taz_work_from_home_base.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
        ],
        'notes': 'Curated lookup list — Base/scripts/Lookup_Lists/',
    },
    # ── Forecast/scripts/Lookup_Lists ─────────────────────────────────────────
    {
        'id': 'DS_LOCAL_006',
        'name': 'Bonus Unit Boundary Parcel Lookup',
        'description': (
            'Lookup mapping parcel APNs to whether they fall within the Bonus Unit Boundary. '
            'Controls eligibility for the jurisdiction bonus unit development pools '
            '(typically CTC asset lands and similar transfer-receiving areas).'
        ),
        'source_path': FCST_LOOKUP_DIR / 'bonus_unit_bndy.csv',
        'dest_name': 'bonus_unit_bndy.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'Curated lookup list — Forecast/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_007',
        'name': 'Known Development Projects',
        'description': (
            'Curated list of known residential, tourist, and commercial development '
            'projects (permitted, under review, or RBU reservations) with unit quantities. '
            'Pre-assigned to specific parcels before the probabilistic allocation step.'
        ),
        'source_path': FCST_LOOKUP_DIR / 'known_projects.csv',
        'dest_name': 'known_projects.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'Curated project list — Forecast/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_008',
        'name': 'Forecast Residential Zoned Units',
        'description': (
            'Target unit counts by jurisdiction and pool type (Bonus Unit, General, ADU, '
            'Town Center, Affordable, etc.) for the 2035 and 2050 forecast years. '
            'This is the core input that drives the parcel allocation model — '
            'every scenario config references these targets.'
        ),
        'source_path': FCST_LOOKUP_DIR / 'forecast_residential_zoned_units.csv',
        'dest_name': 'forecast_residential_zoned_units.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
            'Forecast/Alternative_1/Alternative_1_Forecast.ipynb',
            'Forecast/Alternative_2/Alternative_2_Forecast.ipynb',
            'Forecast/Alternative_3/Alternative_3_Forecast.ipynb',
            'Forecast/Alternative_4/Alternative_4_Forecast.ipynb',
            'Forecast/scripts/Scenario_Forecast.ipynb',
        ],
        'notes': 'Curated target unit table — Forecast/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_009',
        'name': 'Forecast Tourist Assigned Units',
        'description': (
            'Pre-assigned tourist accommodation unit quantities for specific known projects. '
            'Used to bypass the general allocation and directly assign TAUs to parcels.'
        ),
        'source_path': FCST_LOOKUP_DIR / 'forecast_tourist_assigned_units.csv',
        'dest_name': 'forecast_tourist_assigned_units.csv',
        'scripts_used_in': [
            'Forecast/Alternative_3/Alternative_3_Forecast_Reference.ipynb',
        ],
        'notes': 'Curated project list — Forecast/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_010',
        'name': 'Forecast Commercial Assigned Units',
        'description': (
            'Pre-assigned commercial floor area quantities for specific known projects. '
            'Used to directly assign CFA to parcels with approved commercial development.'
        ),
        'source_path': FCST_LOOKUP_DIR / 'forecast_commercial_assigned_units.csv',
        'dest_name': 'forecast_commercial_assigned_units.csv',
        'scripts_used_in': [
            'Forecast/Alternative_3/Alternative_3_Forecast_Reference.ipynb',
        ],
        'notes': 'Curated project list — Forecast/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_011',
        'name': 'CTC Asset Lands Lookup',
        'description': (
            'List of California Tahoe Conservancy (CTC) asset land parcels '
            'identified for special development allocation in the bonus unit pool.'
        ),
        'source_path': FCST_LOOKUP_DIR / 'CTC_AssetLands_Lookup.csv',
        'dest_name': 'CTC_AssetLands_Lookup.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'Curated lookup list — Forecast/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_012',
        'name': 'Forecast Commercial Zoned Units',
        'description': 'Target commercial floor area by jurisdiction and zone type for the forecast period.',
        'source_path': FCST_LOOKUP_DIR / 'forecast_commercial_zoned_units.csv',
        'dest_name': 'forecast_commercial_zoned_units.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'Curated target table — Forecast/scripts/Lookup_Lists/',
    },
    {
        'id': 'DS_LOCAL_013',
        'name': 'Forecast Tourist Zoned Units',
        'description': 'Target tourist accommodation unit counts by jurisdiction and zone type for the forecast period.',
        'source_path': FCST_LOOKUP_DIR / 'forecast_tourist_zoned_units.csv',
        'dest_name': 'forecast_tourist_zoned_units.csv',
        'scripts_used_in': [
            'Forecast/scripts/parcel_attributing.ipynb',
        ],
        'notes': 'Curated target table — Forecast/scripts/Lookup_Lists/',
    },
    # ── Forecast/data/inputs ───────────────────────────────────────────────────
    {
        'id': 'DS_LOCAL_014',
        'name': 'Overnight Visitor Zonal Data — Summer',
        'description': (
            'Base-year overnight visitor counts by TAZ and accommodation type '
            '(hotel/motel, resort, casino, campground, seasonal). Passed through '
            'unchanged to all forecast year outputs.'
        ),
        'source_path': FCST_INPUTS_DIR / 'OvernightVisitorZonalData_Summer.csv',
        'dest_name': 'OvernightVisitorZonalData_Summer_input.csv',
        'scripts_used_in': [
            'Forecast/scripts/run_scenarios.py',
            'Forecast/scripts/Scenario_Forecast.ipynb',
            'Forecast/Alternative_1/Alternative_1_Forecast.ipynb',
            'Forecast/Alternative_2/Alternative_2_Forecast.ipynb',
            'Forecast/Alternative_3/Alternative_3_Forecast.ipynb',
            'Forecast/Alternative_4/Alternative_4_Forecast.ipynb',
        ],
        'notes': 'Input file — Forecast/data/inputs/. Not recomputed by forecast scripts.',
    },
    {
        'id': 'DS_LOCAL_015',
        'name': 'Employment Forecast — 2035',
        'description': (
            'Exogenous employment projections by category (retail, service, recreation, '
            'gaming, other) for the 2035 forecast year by TAZ. '
            'Directly replaces base-year employment in the socioeconomic output.'
        ),
        'source_path': FCST_INPUTS_DIR / 'employment_2035.csv',
        'dest_name': 'employment_2035.csv',
        'scripts_used_in': [
            'Forecast/scripts/run_scenarios.py',
            'Forecast/scripts/Scenario_Forecast.ipynb',
            'Forecast/Alternative_1/Alternative_1_Forecast.ipynb',
            'Forecast/Alternative_2/Alternative_2_Forecast.ipynb',
            'Forecast/Alternative_3/Alternative_3_Forecast.ipynb',
            'Forecast/Alternative_4/Alternative_4_Forecast.ipynb',
        ],
        'notes': 'Exogenous employment forecast — Forecast/data/inputs/',
    },
    {
        'id': 'DS_LOCAL_016',
        'name': 'Employment Forecast — 2050',
        'description': (
            'Exogenous employment projections by category for the 2050 forecast year by TAZ.'
        ),
        'source_path': FCST_INPUTS_DIR / 'employment_2050.csv',
        'dest_name': 'employment_2050.csv',
        'scripts_used_in': [
            'Forecast/scripts/run_scenarios.py',
            'Forecast/scripts/Scenario_Forecast.ipynb',
            'Forecast/Alternative_1/Alternative_1_Forecast.ipynb',
            'Forecast/Alternative_2/Alternative_2_Forecast.ipynb',
            'Forecast/Alternative_3/Alternative_3_Forecast.ipynb',
            'Forecast/Alternative_4/Alternative_4_Forecast.ipynb',
        ],
        'notes': 'Exogenous employment forecast — Forecast/data/inputs/',
    },
    {
        'id': 'DS_LOCAL_017',
        'name': 'Residential Assigned Occupancy / Income Lookup',
        'description': (
            'Lookup table mapping pre-assigned project APNs to specific occupancy rates '
            'and income category proportions. Overrides the general occupancy and income '
            'assignment logic for known projects with documented characteristics.'
        ),
        'source_path': FCST_INPUTS_DIR / 'res_assigned_lookup.csv',
        'dest_name': 'res_assigned_lookup.csv',
        'scripts_used_in': [
            'Forecast/scripts/run_scenarios.py',
            'Forecast/scripts/forecast_functions.py',
        ],
        'notes': 'Curated occupancy/income lookup — Forecast/data/inputs/',
    },
    {
        'id': 'DS_LOCAL_018',
        'name': 'TAZ Town Center Jobs (Base)',
        'description': (
            'Employment multipliers for occupied housing units in town centers. '
            'Applied to add jobs generated by new town center housing to emp_other '
            'in the socioeconomic output (37.5% for 2035, 100% for 2050).'
        ),
        'source_path': FCST_INPUTS_DIR / 'taz_towncenter_jobs.csv',
        'dest_name': 'taz_towncenter_jobs.csv',
        'scripts_used_in': [
            'Forecast/scripts/run_scenarios.py',
            'Forecast/scripts/Job_Distribution.ipynb',
        ],
        'notes': 'Input file — Forecast/data/inputs/',
    },
    {
        'id': 'DS_LOCAL_019',
        'name': 'TAZ Town Center Jobs (Alt 2)',
        'description': 'Alternative 2 variant of town center employment multipliers.',
        'source_path': FCST_INPUTS_DIR / 'taz_towncenter_jobs_alt_2.csv',
        'dest_name': 'taz_towncenter_jobs_alt_2.csv',
        'scripts_used_in': ['Forecast/scripts/run_scenarios.py'],
        'notes': 'Alternative 2 specific — Forecast/data/inputs/',
    },
    {
        'id': 'DS_LOCAL_020',
        'name': 'TAZ Town Center Jobs (Alt 3)',
        'description': 'Alternative 3 variant of town center employment multipliers.',
        'source_path': FCST_INPUTS_DIR / 'taz_towncenter_jobs_alt_3.csv',
        'dest_name': 'taz_towncenter_jobs_alt_3.csv',
        'scripts_used_in': ['Forecast/scripts/run_scenarios.py'],
        'notes': 'Alternative 3 specific — Forecast/data/inputs/',
    },
    {
        'id': 'DS_LOCAL_021',
        'name': 'TAZ Town Center Jobs (Alt 4)',
        'description': 'Alternative 4 variant of town center employment multipliers.',
        'source_path': FCST_INPUTS_DIR / 'taz_towncenter_jobs_alt_4.csv',
        'dest_name': 'taz_towncenter_jobs_alt_4.csv',
        'scripts_used_in': ['Forecast/scripts/run_scenarios.py'],
        'notes': 'Alternative 4 specific — Forecast/data/inputs/',
    },
    {
        'id': 'DS_LOCAL_022',
        'name': 'Forecast Residential Zoned Units (Base Config)',
        'description': (
            'Base version of residential unit allocation targets before '
            'alternative-specific adjustments. Used as the reference starting point '
            'for the lookuplist_data_engineering workflow.'
        ),
        'source_path': FCST_INPUTS_DIR / 'forecast_residential_zoned_units_base.csv',
        'dest_name': 'forecast_residential_zoned_units_base.csv',
        'scripts_used_in': [
            'Forecast/scripts/lookuplist_data_engineering.ipynb',
        ],
        'notes': 'Input file — Forecast/data/inputs/',
    },
    # ── Base/data/raw_data ─────────────────────────────────────────────────────
    {
        'id': 'DS_LOCAL_023',
        'name': 'Occupancy Rates (Raw CSV)',
        'description': (
            'Observed TAU and VHR occupancy rates stored as a local CSV. '
            'Used instead of the live OccupancyRates feature service to ensure '
            'consistency in the IDW interpolation step.'
        ),
        'source_path': BASE_RAW_DIR / 'occupancy_rates.csv',
        'dest_name': 'occupancy_rates_raw.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
        ],
        'notes': 'Raw data file — Base/data/raw_data/',
    },
    {
        'id': 'DS_LOCAL_024',
        'name': 'Census Block Group Rates 2022 (Frozen Snapshot)',
        'description': (
            'Frozen 2022 ACS snapshot of census block group demographic rates — '
            'household occupancy, seasonal residence rate, household size, '
            'and income proportions by block group. '
            'Intentionally stored as a frozen CSV (not re-queried from ACS) to ensure '
            'reproducibility across model runs (see base_year_data_engineering.ipynb Gotcha #11).'
        ),
        'source_path': BASE_RAW_DIR / 'census_block_group_rates_2022.csv',
        'dest_name': 'census_block_group_rates_2022.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
        ],
        'notes': 'Frozen 2022 ACS snapshot — intentionally not re-fetched from live service',
    },
    {
        'id': 'DS_LOCAL_025',
        'name': 'Employment 2022 Data (Raw)',
        'description': 'Raw 2022 employment data by TAZ used as the base year employment input.',
        'source_path': BASE_RAW_DIR / 'employment_2022_data.csv',
        'dest_name': 'employment_2022_data.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
        ],
        'notes': 'Raw data file — Base/data/raw_data/',
    },
    {
        'id': 'DS_LOCAL_026',
        'name': 'Employment Base Year (Processed)',
        'description': (
            'Processed base-year employment by TAZ and category. '
            'Fallback source for the employment processing step in the base year pipeline.'
        ),
        'source_path': BASE_RAW_DIR / 'Employment.csv',
        'dest_name': 'Employment_base_raw.csv',
        'scripts_used_in': [
            'Base/scripts/base_year_data_engineering.ipynb',
        ],
        'notes': 'Raw data file — Base/data/raw_data/',
    },
]

print(f'Local files defined: {len(LOCAL_FILES)}')

ALL_SOURCES = SPATIAL_SERVICES + TABULAR_SERVICES + LTINFO_SOURCES + SQL_SOURCES + LOCAL_FILES
print(f'Total sources      : {len(ALL_SOURCES)}')

Local files defined: 26
Total sources      : 46


## 4 — Create Archive Directories and GDB

In [17]:
# Create output folders if they don't exist
for folder in [ARCHIVE_ROOT, CSV_OUT_DIR, CONFIGS_OUT]:
    folder.mkdir(parents=True, exist_ok=True)
    print(f'  Folder ready: {folder}')

# Create the archive GDB if it doesn't exist
gdb_str = str(GDB_PATH)
if not arcpy.Exists(gdb_str):
    arcpy.management.CreateFileGDB(
        out_folder_path=str(GDB_PATH.parent),
        out_name=GDB_PATH.name
    )
    print(f'  GDB created : {GDB_PATH}')
else:
    print(f'  GDB exists  : {GDB_PATH}')

# Set arcpy environment
arcpy.env.workspace        = gdb_str
arcpy.env.overwriteOutput  = True

# Runtime log — each snapshot step appends its result here
results_log = []

print('\nArchive infrastructure ready.')

  Folder ready: F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM
  Folder ready: F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_Inputs
  Folder ready: F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_Configs
  GDB exists  : F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_DataInventory.gdb

Archive infrastructure ready.


## 5 — Snapshot: Spatial Feature Services → GDB

Each layer is fetched from the TRPA REST service and written as a feature class in `TDM_DataInventory.gdb`.

> **Note on the parcel layer (DS_SPATIAL_001):** This can contain 100k+ features and may take several minutes to fetch. The arcgis SDK handles pagination automatically.

In [18]:
print('=== Snapshotting Spatial Feature Services → GDB ===')
print(f'Target GDB: {GDB_PATH}\n')

for src in SPATIAL_SERVICES:
    snapshot_spatial_to_gdb(src, results_log)

spatial_ok   = sum(1 for r in results_log if r['source_id'].startswith('DS_SPATIAL') and r['status'] == 'SUCCESS')
spatial_fail = sum(1 for r in results_log if r['source_id'].startswith('DS_SPATIAL') and r['status'] != 'SUCCESS')
print(f'\nSpatial services: {spatial_ok}/{len(SPATIAL_SERVICES)} succeeded, {spatial_fail} failed')

=== Snapshotting Spatial Feature Services → GDB ===
Target GDB: F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_DataInventory.gdb

  [DS_SPATIAL_001] Existing Development Parcels
    Query: Year = 2022
    OK — 61,232 features written to Existing_Dev_Parcels
  [DS_SPATIAL_002] Traffic Analysis Zones (TAZ)
    OK — 282 features written to TAZ
  [DS_SPATIAL_003] Census Block Groups


c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthe

    OK — 4,666 features written to Block_Groups
  [DS_SPATIAL_004] Census Tracts
    ERROR: DataFrame must have geometry set.
  [DS_SPATIAL_005] Campgrounds
    Query: RECREATION_TYPE='Campground'
    OK — 18 features written to Campgrounds
  [DS_SPATIAL_006] Tourist Accommodation Occupancy Zones
    OK — 23 features written to Occupancy_Zones
  [DS_SPATIAL_007] Schools (Spatial)
    OK — 21 features written to Schools
  [DS_SPATIAL_008] Town Centers
    OK — 13 features written to Town_Centers

Spatial services: 7/8 succeeded, 1 failed


## 6 — Snapshot: Tabular Feature Services → CSV

In [19]:
print('=== Snapshotting Tabular Feature Services → CSV ===')
print(f'Target folder: {CSV_OUT_DIR}\n')

for src in TABULAR_SERVICES:
    snapshot_tabular_to_csv(src, results_log)

tabular_ok   = sum(1 for r in results_log if r['source_id'].startswith('DS_TABULAR') and r['status'] == 'SUCCESS')
tabular_fail = sum(1 for r in results_log if r['source_id'].startswith('DS_TABULAR') and r['status'] != 'SUCCESS')
print(f'\nTabular services: {tabular_ok}/{len(TABULAR_SERVICES)} succeeded, {tabular_fail} failed')

=== Snapshotting Tabular Feature Services → CSV ===
Target folder: F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_Inputs

  [DS_TABULAR_001] VHR Parcel List


c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthe

    OK — 6,494 records → VHR_List.csv
  [DS_TABULAR_002] Campground Visitation Records
    OK — 18 records → Campground_Visits.csv
  [DS_TABULAR_003] Tourist Accommodation Occupancy Rates (Service)


c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\amcclary\AppData\Local\ESRI\conda\envs\arcgispro-py3-plotly\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'maps.trpa.org'. Adding certificate verification is strongly advised. See: https://urllib3.readthe

    OK — 2,348 records → Occupancy_Rates_Svc.csv
  [DS_TABULAR_004] School Enrollment
    OK — 11 records → School_Enrollment_Svc.csv

Tabular services: 4/4 succeeded, 0 failed


## 7 — Snapshot: LakeTahoeInfo JSON APIs → CSV

In [20]:
print('=== Snapshotting LakeTahoeInfo JSON APIs → CSV ===')
print(f'Target folder: {CSV_OUT_DIR}\n')

for src in LTINFO_SOURCES:
    snapshot_ltinfo_to_csv(src, results_log)

ltinfo_ok   = sum(1 for r in results_log if r['source_id'].startswith('DS_LTINFO') and r['status'] == 'SUCCESS')
ltinfo_fail = sum(1 for r in results_log if r['source_id'].startswith('DS_LTINFO') and r['status'] != 'SUCCESS')
print(f'\nLTInfo sources: {ltinfo_ok}/{len(LTINFO_SOURCES)} succeeded, {ltinfo_fail} failed')

=== Snapshotting LakeTahoeInfo JSON APIs → CSV ===
Target folder: F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_Inputs

  [DS_LTINFO_001] Parcel IPES Scores
    OK — 14,980 records → IPES_Scores_LTInfo.csv
  [DS_LTINFO_002] Parcel Land Capability Values
    OK — 24,655 records → Land_Capability_LTInfo.csv
  [DS_LTINFO_003] All Parcels Status (LTInfo)
    OK — 72,486 records → All_Parcels_LTInfo.csv
  [DS_LTINFO_004] Banked Development Rights
    OK — 2,195 records → Banked_Dev_Rights_LTInfo.csv
  [DS_LTINFO_005] Transacted and Banked Development Rights
    OK — 5,213 records → Transacted_Banked_LTInfo.csv

LTInfo sources: 5/5 succeeded, 0 failed


## 8 — Snapshot: SQL Server Tables → CSV

Requires `DB_USER` and `DB_PASSWORD` environment variables set on this machine.
Geometry columns are automatically excluded using `read_sql_no_geom()`.

In [21]:
print('=== Snapshotting SQL Server Tables → CSV ===')
print(f'Target folder: {CSV_OUT_DIR}\n')

db_user = os.environ.get('DB_USER')
if not db_user:
    print('WARNING: DB_USER environment variable not set. SQL snapshots will fail.')
    print('         Set DB_USER and DB_PASSWORD before running this section.')

for src in SQL_SOURCES:
    snapshot_sql_to_csv(src, results_log)

sql_ok   = sum(1 for r in results_log if r['source_id'].startswith('DS_SQL') and r['status'] == 'SUCCESS')
sql_fail = sum(1 for r in results_log if r['source_id'].startswith('DS_SQL') and r['status'] != 'SUCCESS')
print(f'\nSQL sources: {sql_ok}/{len(SQL_SOURCES)} succeeded, {sql_fail} failed')

=== Snapshotting SQL Server Tables → CSV ===
Target folder: F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_Inputs

  [DS_SQL_001] Permissible Uses (Zoning)
    Query: SELECT * FROM sde.SDE.PermissibleUses
    OK — 10,770 records → PermissibleUses_SQL.csv
  [DS_SQL_002] Special Designation (Zoning)
    Query: SELECT * FROM sde.SDE.Special_Designation
    OK — 515 records → Special_Designation_SQL.csv
  [DS_SQL_003] Parcel History Attributed — 2022
    Query: SELECT * FROM sde.SDE.Parcel_History_Attributed WHERE YEAR = 2022
    OK — 61,232 records → Parcel_History_2022_SQL.csv

SQL sources: 3/3 succeeded, 0 failed


## 9 — Copy Local Curated Files to Archive

In [22]:
print('=== Copying Local Curated Files → Archive ===')
print(f'Target folder: {CSV_OUT_DIR}\n')

for src in LOCAL_FILES:
    copy_local_file(src, results_log)

local_ok   = sum(1 for r in results_log if r['source_id'].startswith('DS_LOCAL') and r['status'] == 'SUCCESS')
local_skip = sum(1 for r in results_log if r['source_id'].startswith('DS_LOCAL') and r['status'] == 'SKIP')
local_fail = sum(1 for r in results_log if r['source_id'].startswith('DS_LOCAL') and r['status'] == 'ERROR')
print(f'\nLocal files: {local_ok} copied, {local_skip} skipped (not found), {local_fail} errors')

=== Copying Local Curated Files → Archive ===
Target folder: F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_Inputs

  [DS_LOCAL_001] Development Changes 2022-2025
    OK — copied to development_2022_2025.csv
  [DS_LOCAL_002] Tourist Accommodation Unit Type Lookup
    OK — copied to lookup_tau_type.csv
  [DS_LOCAL_003] Closed Tourist Parcels
    OK — copied to closed_tourist_parcels.csv
  [DS_LOCAL_004] Income Census Codes
    OK — copied to income_census_codes.csv
  [DS_LOCAL_005] Work-From-Home Rate by TAZ (Base)
    OK — copied to taz_work_from_home_base.csv
  [DS_LOCAL_006] Bonus Unit Boundary Parcel Lookup
    OK — copied to bonus_unit_bndy.csv
  [DS_LOCAL_007] Known Development Projects
    OK — copied to known_projects.csv
  [DS_LOCAL_008] Forecast Residential Zoned Units
    OK — copied to forecast_residential_zoned_units.csv
  [DS_LOCAL_009] Forecast Tourist Assigned Units
    OK — copied to forecast_tourist_assigned_units.csv
  [DS_LOCAL_010] Forecast Commerc

## 10 — Copy Scenario Config JSONs to Archive

In [23]:
print('=== Copying Scenario Config JSONs → Archive ===')
print(f'Source      : {FCST_CONFIGS_DIR}')
print(f'Destination : {CONFIGS_OUT}\n')

config_files = sorted(FCST_CONFIGS_DIR.glob('*.json')) if FCST_CONFIGS_DIR.exists() else []

if not config_files:
    print(f'  No .json files found in {FCST_CONFIGS_DIR}')
else:
    for cfg in config_files:
        dest = CONFIGS_OUT / cfg.name
        shutil.copy2(cfg, dest)
        print(f'  Copied: {cfg.name}')
        sid = f'DS_CONFIG_{cfg.stem}'[:64]
        results_log.append({
            'source_id'       : sid,
            'status'          : 'SUCCESS',
            'record_count'    : 0,
            'archive_location': str(dest),
            'error_message'   : '',
        })
    print(f'\n{len(config_files)} config file(s) copied.')

=== Copying Scenario Config JSONs → Archive ===
Source      : c:\Users\amcclary\Documents\GitHub\Transportation\TravelDemandModel\2026_Housing_EIS\Forecast\configs_final
Destination : F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_Configs

  Copied: alternative_1_04282026_config.json
  Copied: alternative_2_04282026__config.json
  Copied: alternative_3_04282026__config.json

3 config file(s) copied.


## 11 — Build Master Inventory Record

Combines all source metadata with the runtime results into a single `TDM_DataInventory_Record.csv`.

In [24]:
# ── Build lookup of runtime results keyed by source_id ────────────────────────
result_lookup = {r['source_id']: r for r in results_log}


def make_archive_location(src_type, src):
    """Return the expected archive path for a source based on its type."""
    if src_type == 'spatial':
        return os.path.join(str(GDB_PATH), src['fc_name'])
    elif src_type in ('tabular', 'ltinfo'):
        return str(CSV_OUT_DIR / src['csv_name'])
    elif src_type == 'sql':
        return str(CSV_OUT_DIR / src['csv_name'])
    elif src_type == 'local':
        return str(CSV_OUT_DIR / src['dest_name'])
    return ''


def source_to_row(src, src_type):
    """Convert a source definition dict to one inventory record row."""
    sid      = src['id']
    run_res  = result_lookup.get(sid, {})
    scripts  = ' | '.join(src.get('scripts_used_in', []))

    # Determine the canonical URL or path
    if src_type == 'spatial':
        archive_type = 'Feature Class (GDB)'
        url_or_path  = src['url']
        archive_name = src['fc_name']
        query_filter = src.get('query') or ''
    elif src_type == 'tabular':
        archive_type = 'CSV'
        url_or_path  = src['url']
        archive_name = src['csv_name']
        query_filter = src.get('query') or ''
    elif src_type == 'ltinfo':
        archive_type = 'CSV'
        url_or_path  = src['url']
        archive_name = src['csv_name']
        query_filter = ''
    elif src_type == 'sql':
        archive_type = 'CSV'
        url_or_path  = f"SQL Server: sql12/{src['db']}"
        archive_name = src['csv_name']
        query_filter = src['query']
    elif src_type == 'local':
        archive_type = 'CSV (copy)'
        url_or_path  = str(src['source_path'])
        archive_name = src['dest_name']
        query_filter = ''
    else:
        archive_type = 'Unknown'
        url_or_path  = ''
        archive_name = ''
        query_filter = ''

    archive_location = run_res.get('archive_location', make_archive_location(src_type, src))

    return {
        'source_id'          : sid,
        'source_type'        : {
            'spatial' : 'Spatial Feature Service',
            'tabular' : 'Tabular Feature Service',
            'ltinfo'  : 'LakeTahoeInfo JSON API',
            'sql'     : 'SQL Server Table',
            'local'   : 'Local Curated File',
        }.get(src_type, src_type),
        'source_name'        : src['name'],
        'description'        : src.get('description', ''),
        'datasource_url_or_path': url_or_path,
        'query_filter'       : query_filter,
        'scripts_used_in'    : scripts,
        'archive_type'       : archive_type,
        'archive_filename'   : archive_name,
        'archive_full_path'  : archive_location,
        'record_count'       : run_res.get('record_count', ''),
        'status'             : run_res.get('status', 'NOT_RUN'),
        'error_message'      : run_res.get('error_message', ''),
        'snapshot_date'      : SNAPSHOT_DATE,
        'snapshot_timestamp' : SNAPSHOT_TS,
        'notes'              : src.get('notes', ''),
    }


# ── Assemble all rows ─────────────────────────────────────────────────────────
rows = []
for src in SPATIAL_SERVICES:
    rows.append(source_to_row(src, 'spatial'))
for src in TABULAR_SERVICES:
    rows.append(source_to_row(src, 'tabular'))
for src in LTINFO_SOURCES:
    rows.append(source_to_row(src, 'ltinfo'))
for src in SQL_SOURCES:
    rows.append(source_to_row(src, 'sql'))
for src in LOCAL_FILES:
    rows.append(source_to_row(src, 'local'))

# Add config files (dynamic — names discovered at runtime)
for r in results_log:
    sid = r['source_id']
    if sid.startswith('DS_CONFIG_'):
        cfg_name = Path(r['archive_location']).name
        rows.append({
            'source_id'             : sid,
            'source_type'           : 'Scenario Config JSON',
            'source_name'           : f'Scenario Config: {cfg_name}',
            'description'           : (
                'JSON configuration file defining scenario-specific residential unit '
                'targets, zone proportions, occupancy rates, income proportions, and '
                'optional occupancy/population adjustment parameters.'
            ),
            'datasource_url_or_path': str(FCST_CONFIGS_DIR / cfg_name),
            'query_filter'          : '',
            'scripts_used_in'       : 'Forecast/scripts/run_scenarios.py',
            'archive_type'          : 'JSON (copy)',
            'archive_filename'      : cfg_name,
            'archive_full_path'     : r['archive_location'],
            'record_count'          : '',
            'status'                : r['status'],
            'error_message'         : r['error_message'],
            'snapshot_date'         : SNAPSHOT_DATE,
            'snapshot_timestamp'    : SNAPSHOT_TS,
            'notes'                 : 'Scenario config file from Forecast/configs_final/',
        })

df_inventory = pd.DataFrame(rows)
df_inventory.to_csv(INVENTORY_PATH, index=False)

print(f'Inventory written: {INVENTORY_PATH}')
print(f'Total records    : {len(df_inventory)}')
df_inventory[['source_id', 'source_type', 'source_name', 'status', 'record_count']].to_string(index=False)

Inventory written: F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_DataInventory_Record.csv
Total records    : 49


'                               source_id             source_type                                          source_name  status record_count\n                          DS_SPATIAL_001 Spatial Feature Service                         Existing Development Parcels SUCCESS        61232\n                          DS_SPATIAL_002 Spatial Feature Service                         Traffic Analysis Zones (TAZ) SUCCESS          282\n                          DS_SPATIAL_003 Spatial Feature Service                                  Census Block Groups SUCCESS         4666\n                          DS_SPATIAL_004 Spatial Feature Service                                        Census Tracts   ERROR            0\n                          DS_SPATIAL_005 Spatial Feature Service                                          Campgrounds SUCCESS           18\n                          DS_SPATIAL_006 Spatial Feature Service                Tourist Accommodation Occupancy Zones SUCCESS           23\n                   

## 12 — Summary Report

In [25]:
print('=' * 65)
print('  TDM DATA INVENTORY SNAPSHOT — COMPLETE')
print('=' * 65)
print(f'  Snapshot date     : {SNAPSHOT_DATE}')
print(f'  Archive root      : {ARCHIVE_ROOT}')
print(f'  GDB               : {GDB_PATH}')
print(f'  CSV / local files : {CSV_OUT_DIR}')
print(f'  Config JSONs      : {CONFIGS_OUT}')
print(f'  Inventory record  : {INVENTORY_PATH}')
print()

# Status by source type
if len(df_inventory) > 0:
    summary = (
        df_inventory
        .groupby(['source_type', 'status'])
        .size()
        .unstack(fill_value=0)
    )
    print('Status by source type:')
    print(summary.to_string())
    print()

    total     = len(df_inventory)
    succeeded = (df_inventory['status'] == 'SUCCESS').sum()
    skipped   = (df_inventory['status'] == 'SKIP').sum()
    failed    = (df_inventory['status'] == 'ERROR').sum()
    not_run   = (df_inventory['status'] == 'NOT_RUN').sum()

    print(f'Overall: {succeeded}/{total} succeeded  |  {skipped} skipped  |  {failed} errors  |  {not_run} not run')

    if failed > 0:
        print()
        print('FAILED SOURCES:')
        failed_rows = df_inventory[df_inventory['status'] == 'ERROR'][['source_id', 'source_name', 'error_message']]
        for _, row in failed_rows.iterrows():
            print(f'  [{row["source_id"]}] {row["source_name"]}')
            print(f'    {row["error_message"][:120]}')

print()
print('Archive contents:')
print(f'  GDB feature classes : {len([r for r in results_log if r["status"] == "SUCCESS" and r["source_id"].startswith("DS_SPATIAL")])}')
csv_count = len(list(CSV_OUT_DIR.glob('*.csv')))
json_count = len(list(CONFIGS_OUT.glob('*.json')))
print(f'  CSVs in TDM_Inputs  : {csv_count}')
print(f'  JSONs in TDM_Configs: {json_count}')
print('=' * 65)

  TDM DATA INVENTORY SNAPSHOT — COMPLETE
  Snapshot date     : 2026-05-12
  Archive root      : F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM
  GDB               : F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_DataInventory.gdb
  CSV / local files : F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_Inputs
  Config JSONs      : F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_Configs
  Inventory record  : F:\GIS\GIS_DATA\AnalysisDataArchives\CultivatingCommunityTDM\TDM_DataInventory_Record.csv

Status by source type:
status                   ERROR  SUCCESS
source_type                            
LakeTahoeInfo JSON API       0        5
Local Curated File           0       26
SQL Server Table             0        3
Scenario Config JSON         0        3
Spatial Feature Service      1        7
Tabular Feature Service      0        4

Overall: 48/49 succeeded  |  0 skipped  |  1 errors  |  0 not run

FAILED SOURCES:
  [DS_SPA